# Day 4 - Medal Prediction Model Training

This notebook trains baseline machine learning models to predict whether an athlete wins a medal using SQL-engineered features from PostgreSQL.

## Imports and Config

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sqlalchemy import create_engine

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "won_medal"

PROJECT_ROOT = Path.cwd().resolve().parent
MODEL_DIR = PROJECT_ROOT / "src" / "models" / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifact dir: {MODEL_DIR}")

Project root: /Users/dylando/Desktop/Personal Coding/olympic-medal-prediction
Model artifact dir: /Users/dylando/Desktop/Personal Coding/olympic-medal-prediction/src/models/artifacts


## Load Modeling Dataset from PostgreSQL

In [2]:
engine = create_engine("postgresql://localhost:5432/olympics_db")

df = pd.read_sql("SELECT * FROM model_dataset", engine)

print("Loaded model dataset from PostgreSQL")
print(f"Shape: {df.shape}")

df.head()

Loaded model dataset from PostgreSQL
Shape: (8500, 8)


,athlete_id,sport,event,team_or_individual,medal,country_strength,sport_medal_rate,athlete_experience
0,ATH-00001,Rowing,Four W,Team,No Medal,0.205,0.263,1
1,ATH-00002,Ski Jumping,Normal Hill Team,Individual,Silver,0.226,0.270,1
2,ATH-00003,Figure Skating,Women's Singles,Individual,No Medal,0.265,0.271,1
3,ATH-00004,Triathlon,Men's Triathlon,Individual,No Medal,0.329,0.223,1
4,ATH-00005,Triathlon,Men's Triathlon,Individual,No Medal,0.299,0.223,1


## Target and Feature Prep

In [3]:
df[TARGET] = (df["medal"] != "No Medal").astype(int)

feature_cols = [
    "sport",
    "event",
    "team_or_individual",
    "country_strength",
    "sport_medal_rate",
    "athlete_experience",
]

X = df[feature_cols]
y = df[TARGET]

print("Target distribution:")
print(y.value_counts(normalize=True).rename("proportion"))

print(f"Feature matrix shape: {X.shape}")

Target distribution:
won_medal
0    0.763647
1    0.236353
Name: proportion, dtype: float64
Feature matrix shape: (8500, 6)


## Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (6800, 6)
Test shape: (1700, 6)


## Build Preprocessing Pipeline

In [5]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

Numeric features: ['country_strength', 'sport_medal_rate', 'athlete_experience']
Categorical features: ['sport', 'event', 'team_or_individual']


## Train Logistic Regression

In [6]:
logreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

logreg_model.fit(X_train, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


## Evaluate Logistic Regression

In [7]:
y_pred_logreg = logreg_model.predict(X_test)
y_prob_logreg = logreg_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_logreg, digits=3))

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_logreg))

print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_test, y_prob_logreg):.4f}")

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0      0.798     0.524     0.633      1298
           1      0.271     0.572     0.368       402

    accuracy                          0.535      1700
   macro avg      0.535     0.548     0.500      1700
weighted avg      0.674     0.535     0.570      1700

Logistic Regression Confusion Matrix:
[[680 618]
 [172 230]]
Logistic Regression ROC-AUC: 0.5640


## Train Random Forest

In [8]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

rf_model.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


## Evaluate Random Forest

In [9]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, digits=3))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print(f"Random Forest ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

Random Forest Classification Report:
              precision    recall  f1-score   support

           0      0.783     0.767     0.775      1298
           1      0.294     0.313     0.304       402

    accuracy                          0.660      1700
   macro avg      0.539     0.540     0.539      1700
weighted avg      0.667     0.660     0.664      1700

Random Forest Confusion Matrix:
[[996 302]
 [276 126]]
Random Forest ROC-AUC: 0.5581


## Compare Models

In [10]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_rf),
    ],
    "Precision": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_rf),
    ],
    "Recall": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_rf),
    ],
    "F1": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_rf),
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, y_prob_logreg),
        roc_auc_score(y_test, y_prob_rf),
    ],
})

results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.535294,0.271226,0.572139,0.368000,0.563967
1,Random Forest,0.660000,0.294393,0.313433,0.303614,0.558069


## Model Interpretation

The baseline models achieved modest predictive performance, with ROC-AUC scores slightly above random classification.

Key observations:
- Logistic Regression achieved higher recall, identifying more medal-winning athletes but producing more false positives.
- Random Forest achieved higher overall accuracy by favoring the majority "No Medal" class.
- The relatively low ROC-AUC scores suggest that additional feature engineering and more informative historical performance features are needed to improve medal prediction performance.

These results establish a baseline for future experimentation and model improvement.

## Save Best Model Artifact

In [11]:
best_model = rf_model if roc_auc_score(y_test, y_prob_rf) >= roc_auc_score(y_test, y_prob_logreg) else logreg_model
best_model_name = "random_forest" if best_model == rf_model else "logistic_regression"

model_path = MODEL_DIR / f"medal_prediction_{best_model_name}.joblib"

joblib.dump(best_model, model_path)

print(f"Saved best model to: {model_path}")

Saved best model to: /Users/dylando/Desktop/Personal Coding/olympic-medal-prediction/src/models/artifacts/medal_prediction_logistic_regression.joblib


## Day 4 Model Training Summary

This notebook trained baseline medal prediction models using SQL-engineered features from PostgreSQL.

Key takeaways:
- Medal prediction is an imbalanced classification problem because most athletes do not win medals.
- Logistic Regression provides an interpretable baseline.
- Random Forest captures non-linear relationships between country strength, sport medal rates, athlete experience, and medal outcomes.
- ROC-AUC, precision, recall, and F1 are more meaningful than accuracy alone for this problem.

Next steps:
- Add stronger time-aware features to reduce data leakage.
- Tune model hyperparameters.
- Compare with XGBoost.
- Add model interpretability using feature importance or SHAP.